### Environment Setup

#### Virtual Environment

Create and activate a virtual environment:

```bash
python -m venv venv
source venv/bin/activate  # On Windows: venv\Scripts\activate
```

#### Install Dependencies

Install required packages:

```bash
pip install -r requirements.txt
```

### Import Required Libraries

Import all necessary libraries for data processing, model training, and evaluation.

In [ ]:
import torch
import numpy as np
from datasets import Dataset, ClassLabel
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

/usr/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Dataset

Load the Tamil abuse detection dataset from CSV file.

In [ ]:
from datasets import load_dataset
dataset = load_dataset(
"csv",
data_files="trainV2.csv"
)


---
#### Data Preprocessing

Normalize the data by:
- Removing whitespace at the end of labels
- Converting labels to consistent case (handling variations like "abusive" vs "Abusive")
---

In [ ]:
label2id = {
    "Non-Abusive": 0,
    "Abusive": 1
}
def encode_labels(example):
    label = example["Class"].strip()
    if label.lower() == "abusive":
        example["Class"] = label2id["Abusive"]
    elif label.lower() == "non-abusive":
        example["Class"] = label2id["Non-Abusive"]
    else:
        # Fallback: try direct lookup
        example["Class"] = label2id[label]
    return example
dataset = dataset.map(encode_labels)

#### Rename Columns

Rename columns to match the expected format for model training:
- `Text` → `text`
- `Class` → `label`

In [ ]:
dataset = dataset.rename_column("Text", "text")
dataset = dataset.rename_column("Class", "label")

#### Rename Columns
- Encoding string labels to integer IDs (0: Non-Abusive, 1: Abusive) 

In [ ]:
def convert_label(example):
    example["label"] = int(example["label"])
    return example

dataset = dataset.map(convert_label)


In [ ]:
dataset["train"][:5]

#### Cast to ClassLabel for Stratification

convert the label from char array to int array

In [ ]:
from datasets import ClassLabel
dataset = dataset.cast_column("label", ClassLabel(num_classes=2, names=["Non-Abusive", "Abusive"]))


In [ ]:
dataset.keys()

In [ ]:
dataset["train"][:5]

### Dataset Statistics

Analyze the label distribution in the dataset to understand class balance.

In [ ]:
from collections import Counter

# Count labels in the full dataset
label_counts = Counter(dataset["train"]["label"])
print("Dataset Label Distribution:")
print(f"Non-Abusive (0): {label_counts.get(0, 0)}")
print(f"Abusive (1): {label_counts.get(1, 0)}")
print(f"Total: {sum(label_counts.values())}")

### Train/Test Split

Split the dataset into training (80%) and test (20%) sets with stratification to maintain class distribution in both splits.

In [ ]:
dataset_split = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)
train_dataset = dataset_split["train"]
test_dataset = dataset_split["test"]

### Training Set Statistics

Analyze the label distribution in the training set.

In [ ]:
# Count labels in training set
train_label_counts = Counter(train_dataset["label"])
print("Training Set Label Distribution:")
print(f"Non-Abusive (0): {train_label_counts.get(0, 0)}")
print(f"Abusive (1): {train_label_counts.get(1, 0)}")
print(f"Total: {sum(train_label_counts.values())}")

### Test Set Statistics

Analyze the label distribution in the test set.

In [ ]:
# Count labels in test set
test_label_counts = Counter(test_dataset["label"])
print("Test Set Label Distribution:")
print(f"Non-Abusive (0): {test_label_counts.get(0, 0)}")
print(f"Abusive (1): {test_label_counts.get(1, 0)}")
print(f"Total: {sum(test_label_counts.values())}")

#### Tokenization for XLM-RoBERTa

Tokenize the Tamil text using the `xlm-roberta-base` tokenizer.

In [ ]:
model_name = "xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)
model = XLMRobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Non-Abusive", 1: "Abusive"},
    label2id={"Non-Abusive": 0, "Abusive": 1},
)

# Tokenization function
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Apply tokenization to train and test datasets
encoded_train = train_dataset.map(tokenize_function, batched=True)
encoded_test = test_dataset.map(tokenize_function, batched=True)

# Set torch format for Trainer
encoded_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
encoded_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

### Training XLM-RoBERTa

Configure training arguments and train the classifier.

In [ ]:
# Metrics for evaluation

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

# Training arguments
training_args = TrainingArguments(
    output_dir="./xlmr-tamil-abuse",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_test,
    compute_metrics=compute_metrics,
)
trainer.train()

#### Evaluation

Evaluate the trained model on the test set.

In [ ]:
# Evaluate on test set
eval_results = trainer.evaluate(encoded_test)
print(eval_results)

#### Prediction on New Text

Use the fine-tuned model to predict whether new Tamil comments are abusive or non-abusive.

In [ ]:
def predict(texts):
    """Predict labels for a list of Tamil texts."""
    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt")
    with torch.no_grad():
        outputs = model(**encodings)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
    id2label = {0: "Non-Abusive", 1: "Abusive"}
    return [id2label[int(p)] for p in preds]

example_texts = [
    "நான் உன்னை மிகவும் விரும்புகிறேன்",  # likely Non-Abusive
    "உனக்கு வெட்கமா இல்லையா"            # potentially Abusive depending on context
]
print(predict(example_texts))